# Results and mechanism analysis

Run after notebooks 01 and 02. Read their explicitly registered results, reproduce the report evidence, and distinguish prediction agreement, fault detection and limitations. No model is trained here.


## 1  Imports and consistent names

Register six detectors separately from the four distinct predictors. CUSUM statistics are not CO2 predictions.

In [ ]:

# Find the project from its root or the notebooks directory.
from pathlib import Path
import sys
_here=Path.cwd().resolve()
PROJECT_DIR=next((p for p in (_here,*_here.parents) if (p/'src/co2_project').is_dir()),None)
if PROJECT_DIR is None: raise FileNotFoundError('Open this notebook inside the submission project.')
if str(PROJECT_DIR/'src') not in sys.path: sys.path.insert(0,str(PROJECT_DIR/'src'))
from co2_project.project import load_settings, resolve_data_csv, record_run, get_run
from co2_project.data import file_sha256, prepare_environment_csv, make_features, chronological_split, HISTORY_FEATURES
from co2_project.faults import WindowPlanner, write_scenario, inject_fault
from co2_project.models import fit_ols, predict_ols_features, fit_neural, neural_settings
from co2_project.detectors import cusum_series, normalized_cusum, select_cusum_limit
from co2_project.evaluation import evaluate_alarm_events, score_development_events
from functools import partial
SETTINGS=load_settings(PROJECT_DIR)
cusum_series=partial(cusum_series,max_gap_minutes=SETTINGS['MAX_GAP_MINUTES'])
normalized_cusum=partial(normalized_cusum,max_gap_minutes=SETTINGS['MAX_GAP_MINUTES'])
select_cusum_limit=partial(select_cusum_limit,max_gap_minutes=SETTINGS['MAX_GAP_MINUTES'])

# Imports and consistent names
from pathlib import Path
import hashlib
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import timezone
from IPython.display import display, Markdown

METHODS = ['change_rule', 'history_regression', 'history_tree', 'history_neural',
           'history_neural_cusum', 'nn_threshold_cusum']
PREDICTORS = METHODS[:4]
FOCUS_METHODS = ['history_neural', 'history_neural_cusum', 'nn_threshold_cusum']
METHOD_NAMES = dict(zip(METHODS, ['Adjacent change rule', 'OLS', 'Regression tree', 'NN point',
                                 'Original NN + CUSUM', 'Selected NN + CUSUM']))
PREDICTOR_NAMES = {**METHOD_NAMES, 'change_rule': 'Persistence', 'history_neural': 'NN'}
COLOURS = dict(zip(METHODS, ['#8c564b', '#1f77b4', '#2a8c57', '#8357a8', '#d47b24', '#157c83']))
plt.rcParams.update({'font.size': 10, 'figure.dpi': 110,
                     'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 80)

from datetime import datetime


## 2  Load the pinned completed experiment

Training, calibration and injection were completed in the original experiment. This cell reads their tables and outputs. Analysis exports contain course measurements and remain local.

In [ ]:
# Load the pinned completed experiment
# Read only from the pinned experiment. Write only to a separate analysis folder.

RUN_DIR=get_run(PROJECT_DIR,'main')
INPUT_RUN_DIR=get_run(PROJECT_DIR,'input_comparison')
input_report=json.loads((INPUT_RUN_DIR/'run_report.json').read_text(encoding='utf-8'))
if (PROJECT_DIR/input_report['retained_baseline']).resolve()!=RUN_DIR.resolve():
    raise ValueError('Input comparison belongs to another main run. Rerun notebook 02.')
if input_report['audit']['prepared_sha256']!=json.loads((RUN_DIR/'run_report.json').read_text(encoding='utf-8'))['audit']['prepared_sha256']:
    raise ValueError('The registered experiments use different prepared sensor data. Rerun notebooks 01 and 02.')
EXPORT_DIR=PROJECT_DIR/'results/analysis'/datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
SAVE_OUTPUTS=True
DECOMPOSITION_FILE=INPUT_RUN_DIR/'prediction_shift_readings.csv'
REQUIRED = ['run_report.json', 'frozen_selection.json', 'prediction_quality.csv',
            'comparison_all_methods.csv', 'thresholds.csv', 'event_results.csv',
            'test_sensitivity.csv', 'test_sensitivity_events.csv',
            'test_sensitivity_aggregate.csv', 'timing_summary.csv']
missing = [name for name in REQUIRED if not (RUN_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Pinned experiment is incomplete: {missing}. Check RUN_DIR in this cell.')
if EXPORT_DIR.resolve() == RUN_DIR.resolve() or RUN_DIR.resolve() in EXPORT_DIR.resolve().parents:
    raise ValueError('EXPORT_DIR must be outside the source experiment.')

def file_hash(path):
    """Read a file hash without modifying its contents."""
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

SOURCE_FILES = sorted(p for p in RUN_DIR.rglob('*') if p.is_file())
SOURCE_HASHES = {str(p.relative_to(RUN_DIR)): file_hash(p) for p in SOURCE_FILES}
report = json.loads((RUN_DIR / 'run_report.json').read_text(encoding='utf-8'))
selection = json.loads((RUN_DIR / 'frozen_selection.json').read_text(encoding='utf-8'))
quality = pd.read_csv(RUN_DIR / 'prediction_quality.csv')
comparison = pd.read_csv(RUN_DIR / 'comparison_all_methods.csv')
thresholds = pd.read_csv(RUN_DIR / 'thresholds.csv')
events = pd.read_csv(RUN_DIR / 'event_results.csv')
sensitivity = pd.read_csv(RUN_DIR / 'test_sensitivity.csv')
sensitivity_events = pd.read_csv(RUN_DIR / 'test_sensitivity_events.csv')
saved_aggregate = pd.read_csv(RUN_DIR / 'test_sensitivity_aggregate.csv')
timing = pd.read_csv(RUN_DIR / 'timing_summary.csv')
manifest = {kind: json.loads((RUN_DIR / f'fault_data/{kind}/manifest.json').read_text(encoding='utf-8'))
            for kind in ['spike', 'drift', 'bias']}

def save_table(frame, filename, index=False):
    """Export an analysis table, leaving all source files unchanged."""
    if SAVE_OUTPUTS:
        EXPORT_DIR.mkdir(parents=True, exist_ok=True)
        frame.to_csv(EXPORT_DIR / filename, index=index, encoding='utf-8-sig')

def show_figure(fig, filename):
    """Save a new scientific plot and display it in the notebook."""
    if SAVE_OUTPUTS:
        EXPORT_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(EXPORT_DIR / filename, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(fig)

print('Read experiment:', RUN_DIR)
print('Study:', report['study_version'], '|', report['status'])
print('Sensor:', report['audit']['sensor_id'], '| split counts:', report['split_counts'])
print('NN fitted on:', report['models']['history_neural']['fit_on'])
print('Analysis exports:', EXPORT_DIR)


## 3  Organise scope and eight explanatory variables

Inputs stay fixed. Magnitude and duration are study factors; fault seeds check placement sensitivity. The completed six-input NN trial is read from notebook 02.

In [ ]:
# Organise scope and eight explanatory variables
feature_names = report['models']['history_neural']['features']
reasons = {
    'co2_lag1': 'Recent continuity; a fault can enter the next prediction.',
    'co2_lag6': 'Earlier history; six readings do not guarantee one hour.',
    'temperature': 'Environmental context, not independent CO2 truth.',
    'humidity': 'Environmental context, not independent CO2 truth.',
    'hour_sin': 'Cyclical hour in UTC; an activity proxy only.',
    'hour_cos': 'Completes the cyclical UTC hour representation.',
    'weekend': 'Weekly activity proxy; actual occupancy is unobserved.',
    'minutes_since_previous': 'Irregular sampling context.'}
feature_table = pd.DataFrame({'feature': feature_names,
                              'reason_and_limit': [reasons[f] for f in feature_names]})
scope = pd.DataFrame([
    ['Main comparison', 'Six detectors; spike, drift and bias', 'Retain all existing methods'],
    ['Detailed mechanism', 'NN residuals and two CUSUM settings', 'Explain detection and misses'],
    ['Fixed inputs', 'Eight shared variables', 'No feature search in this notebook'],
    ['Fault factors', '20/40/60 ppm; long faults 12/24/48 readings', 'Sensitivity, not training inputs'],
    ['Fault seeds', '7/42/99', 'All retained; do not pick the best seed'],
    ['Model seed', '42', 'Fixed training seed, distinct from fault placement'],
    ['Evidence', 'One sensor and an inspected test period', 'Exploratory simulated-fault findings']],
    columns=['role', 'setting', 'decision'])
display(scope)
display(feature_table)
save_table(scope, 'scope.csv')
save_table(feature_table, 'input_variables.csv')


## 4  Check scenario counts and consistent versions

Expect 9 spike, 27 drift and 27 bias settings, giving 378 detector rows. The three NN detectors must share predictions. These checks concern implementation consistency, not real-fault validity.

In [ ]:
# Check scenario counts and consistent versions
case_keys = ['scenario', 'seed', 'magnitude_ppm', 'duration_readings']
cases = sensitivity[case_keys].drop_duplicates()
assert not sensitivity.duplicated(case_keys + ['method']).any()
assert set(sensitivity.method) == set(METHODS)
assert len(cases) == 63 and len(sensitivity) == 378
assert cases.groupby('scenario').size().to_dict() == {'bias': 27, 'drift': 27, 'spike': 9}
assert set(sensitivity.seed) == {7, 42, 99}
assert report['split_counts'] == {'train': 7373, 'validation': 2458, 'test': 2458, 'excluded': 1511}
assert not events.duplicated(['scenario', 'method', 'event_id']).any()
assert (sensitivity.paired_additional_events <= sensitivity.detected).all()
assert (sensitivity.detected <= sensitivity.events).all()
assert set(cases.magnitude_ppm) == {20.0, 40.0, 60.0}
point_cache = {}

def load_points(kind, method):
    """Read a saved default replay; return a copy so plotting cannot alter cached values."""
    key = (kind, method)
    if key not in point_cache:
        p = pd.read_csv(RUN_DIR / f'results/{kind}/{method}_points.csv', index_col=0, parse_dates=True)
        p.index = pd.to_datetime(p.index, utc=True)
        assert p.index.is_unique and p.index.is_monotonic_increasing
        assert len(p) == report['split_counts']['test']
        assert pd.api.types.is_bool_dtype(p.alarm)
        point_cache[key] = p
    return point_cache[key].copy()

for kind in ['spike', 'drift', 'bias']:
    nn = load_points(kind, 'history_neural')
    for method in FOCUS_METHODS[1:]:
        other = load_points(kind, method)
        pd.testing.assert_index_equal(nn.index, other.index)
        np.testing.assert_allclose(nn.prediction_ppm, other.prediction_ppm, rtol=0, atol=1e-10)
    for method in METHODS:
        p = load_points(kind, method)
        pd.testing.assert_index_equal(nn.index, p.index)
        np.testing.assert_allclose(p.observed_co2_ppm, nn.observed_co2_ppm, rtol=0, atol=1e-10)
case_counts = cases.groupby('scenario').size().rename('settings').to_frame()
display(case_counts)
print('All six detectors receive matching observations. Three NN alarm rules share predictions.')
save_table(case_counts, 'case_counts.csv', index=True)


## 5  Compare prediction agreement with unmodified records

MAE is mean absolute difference; RMSE weights larger differences more strongly. Both use ppm and reference an unverified original record. Only four distinct predictions are shown.

In [ ]:
# Compare prediction agreement with unmodified records
q = quality[quality.part.isin(['validation', 'test_unmodified'])].copy()
prediction_table = q.pivot(index='method', columns='part', values=['mae_ppm', 'rmse_ppm']).reindex(PREDICTORS)
prediction_table.columns = [f'{metric}_{part}' for metric, part in prediction_table.columns]
prediction_table.index = [PREDICTOR_NAMES[m] for m in prediction_table.index]
display(prediction_table.round(3))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), layout='constrained')
for ax, metric in zip(axes, ['mae_ppm', 'rmse_ppm']):
    view = q.pivot(index='method', columns='part', values=metric).reindex(PREDICTORS)
    x = np.arange(len(view))
    ax.bar(x - .18, view['validation'], .36, label='Validation')
    ax.bar(x + .18, view['test_unmodified'], .36, label='Unmodified test')
    ax.set_xticks(x, [PREDICTOR_NAMES[m] for m in view.index], rotation=15)
    ax.set_ylabel('ppm')
    ax.set_title(metric.replace('_ppm', '').upper())
    ax.set_ylim(0, 1.25 * float(view.to_numpy().max()))
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=.2)
show_figure(fig, 'prediction_agreement.png')
save_table(prediction_table, 'prediction_agreement.csv', index=True)


## 6  Compare all six detectors under the default faults

Default seed 42 and 60 ppm; spikes last one reading, drift/bias last 24. An event needs one active-interval alarm. A paired additional event needs an injected alarm at a timestamp without an unmodified alarm; the entire original window need not be quiet.

In [ ]:
# Compare all six detectors under the default faults
default_table = pd.DataFrame(index=METHODS)
for kind in ['spike', 'drift', 'bias']:
    k = comparison[comparison.scenario == kind].set_index('method').reindex(METHODS)
    default_table[kind] = [f'{int(a)}/{int(b)}' for a, b in zip(k.detected, k.events)]
    default_table[kind + '_paired_additional'] = [f'{int(a)}/{int(b)}' for a, b in zip(k.paired_additional_events, k.events)]
background = comparison[comparison.scenario == 'drift'].set_index('method').reindex(METHODS)
default_table['background_alerts'] = [f'{round(v * 2458)}/2458' for v in background.unmodified_test_alert_rate]
default_table.index = [METHOD_NAMES[m] for m in default_table.index]
display(default_table)
threshold_table = thresholds.set_index('method').reindex(METHODS)[['threshold', 'validation_alert_rate']].copy()
threshold_table['unit'] = ['ppm absolute change', 'ppm absolute residual', 'ppm absolute residual',
                           'ppm absolute residual', 'accumulated ppm residual', 'dimensionless accumulated score']
threshold_table.index = [METHOD_NAMES[m] for m in threshold_table.index]
display(threshold_table.round(4))
print('Background alerts are not verified false positives. Threshold units differ across alarm rules.')
save_table(default_table, 'default_comparison.csv', index=True)
save_table(threshold_table, 'thresholds_with_units.csv', index=True)


## 7  Pool three seeds at a fixed magnitude and duration

This is the overall comparison at common conditions. All three fault seeds are retained; spikes always last one reading. After changing conditions, rerun subsequent analyses.

In [ ]:
# Pool three seeds at a fixed magnitude and duration
MAIN_MAGNITUDE_PPM = 60.0   # Existing choices: 20, 40, 60.
MAIN_DURATION_READINGS = 24 # Long-fault choices: 12, 24, 48; spikes always use 1.
if MAIN_MAGNITUDE_PPM not in set(sensitivity.magnitude_ppm):
    raise ValueError('Choose a magnitude already present in the saved experiment.')
if MAIN_DURATION_READINGS not in {12, 24, 48}:
    raise ValueError('Long-fault duration must be 12, 24 or 48 readings.')
common = sensitivity[(sensitivity.magnitude_ppm == MAIN_MAGNITUDE_PPM) &
                     (((sensitivity.scenario == 'spike') & (sensitivity.duration_readings == 1)) |
                      ((sensitivity.scenario != 'spike') & (sensitivity.duration_readings == MAIN_DURATION_READINGS)))]
pooled = common.groupby(['scenario', 'method']).agg(
    detected=('detected', 'sum'), events=('events', 'sum'),
    paired_additional=('paired_additional_events', 'sum'), seeds=('seed', 'nunique'))
assert (pooled.seeds == 3).all()
pooled_table = pd.DataFrame(index=METHODS)
for kind in ['spike', 'drift', 'bias']:
    part = pooled.loc[kind].reindex(METHODS)
    pooled_table[kind] = [f'{int(a)}/{int(b)}' for a,b in zip(part.detected, part.events)]
    pooled_table[kind + '_paired_additional'] = [f'{int(a)}/{int(b)}' for a,b in zip(part.paired_additional, part.events)]
pooled_table.index = [METHOD_NAMES[m] for m in pooled_table.index]
print('Magnitude:', MAIN_MAGNITUDE_PPM, 'ppm | long-fault duration:', MAIN_DURATION_READINGS, 'readings')
display(pooled_table)
print('Pooled events reuse the same measured period; they are not independent real faults.')
save_table(pooled_table, 'three_seed_common_conditions.csv', index=True)


## 8  Inspect differences between fault seeds

Seeds change placement and sign on the same record. Show all six methods in the table and the three focus methods in the plot. Seed ranges are not confidence intervals.

In [ ]:
# Inspect differences between fault seeds
drift_common = common[common.scenario == 'drift'].copy()
seeds = sorted(drift_common.seed.unique())
seed_numeric = drift_common.pivot(index='method', columns='seed', values='detected').reindex(METHODS)
seed_denominators = drift_common.pivot(index='method', columns='seed', values='events').reindex(METHODS)
seed_table = seed_numeric.copy().astype(object)
for seed in seeds:
    seed_table[seed] = [f'{int(a)}/{int(b)}' for a,b in zip(seed_numeric[seed], seed_denominators[seed])]
seed_table['pooled'] = [f'{int(seed_numeric.loc[m].sum())}/{int(seed_denominators.loc[m].sum())}' for m in METHODS]
seed_table.index = [METHOD_NAMES[m] for m in seed_table.index]
display(seed_table)
fig, ax = plt.subplots(figsize=(9, 3.8), layout='constrained')
for method in FOCUS_METHODS:
    rates = 100 * seed_numeric.loc[method] / seed_denominators.loc[method]
    ax.plot(range(len(seeds)), rates, marker='o', label=METHOD_NAMES[method], color=COLOURS[method])
ax.set_xticks(range(len(seeds)), [str(s) for s in seeds])
ax.set(xlabel='Fault placement seed', ylabel='Drift event detection (%)', ylim=(-3, 103),
       title=f'Drift at {MAIN_MAGNITUDE_PPM:g} ppm over {MAIN_DURATION_READINGS} readings')
ax.legend(fontsize=9)
ax.grid(alpha=.2)
show_figure(fig, 'drift_seed_variation.png')
save_table(seed_table, 'drift_by_seed.csv', index=True)


## 9  Analyse drift magnitude and duration

The focus figure shows NN point detection and both CUSUM settings; the complete six-method grid is exported. Set HEATMAP_METHODS to METHODS.copy() for all six. METRIC selects raw or paired additional detections. Duration can change eligible positions, so this is overall sensitivity rather than an isolated causal experiment.

In [ ]:
# Analyse drift magnitude and duration
HEATMAP_METHODS = FOCUS_METHODS.copy()
METRIC = 'detected' # Alternative: 'paired_additional_events'.
if not HEATMAP_METHODS or not set(HEATMAP_METHODS).issubset(METHODS):
    raise ValueError('HEATMAP_METHODS must contain saved detector keys.')
if METRIC not in {'detected', 'paired_additional_events'}:
    raise ValueError('Choose detected or paired_additional_events.')
aggregate = sensitivity.groupby(['scenario', 'method', 'magnitude_ppm', 'duration_readings'], as_index=False).agg(
    runs=('seed', 'nunique'), events=('events', 'sum'), detected=('detected', 'sum'),
    paired_additional_events=('paired_additional_events', 'sum'))
aggregate_keys = ['scenario', 'method', 'magnitude_ppm', 'duration_readings']
compare_columns = aggregate_keys + ['runs', 'events', 'detected', 'paired_additional_events']
pd.testing.assert_frame_equal(
    aggregate[compare_columns].sort_values(aggregate_keys).reset_index(drop=True),
    saved_aggregate[compare_columns].sort_values(aggregate_keys).reset_index(drop=True), check_dtype=False)
full_drift = aggregate[aggregate.scenario == 'drift'].copy()
full_drift['count'] = [f'{int(a)}/{int(b)}' for a,b in zip(full_drift[METRIC], full_drift.events)]
drift_grid_table = full_drift.pivot(index=['magnitude_ppm', 'duration_readings'], columns='method', values='count').reindex(columns=METHODS)
drift_grid_table.columns = [METHOD_NAMES[m] for m in drift_grid_table.columns]
display(drift_grid_table)
ncols = min(3, len(HEATMAP_METHODS))
nrows = math.ceil(len(HEATMAP_METHODS) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3.6*nrows), layout='constrained', squeeze=False)
for ax, method in zip(axes.flat, HEATMAP_METHODS):
    part = full_drift[full_drift.method == method]
    counts = part.pivot(index='magnitude_ppm', columns='duration_readings', values=METRIC).sort_index()
    totals = part.pivot(index='magnitude_ppm', columns='duration_readings', values='events').reindex(index=counts.index, columns=counts.columns)
    fractions = counts / totals
    ax.imshow(fractions, vmin=0, vmax=1, cmap='Blues', aspect='auto')
    ax.set_xticks(range(len(counts.columns)), [str(int(v)) for v in counts.columns])
    ax.set_yticks(range(len(counts.index)), [f'{v:g}' for v in counts.index])
    ax.set(xlabel='Duration in readings', ylabel='Final magnitude in ppm', title=METHOD_NAMES[method])
    for i in range(len(counts.index)):
        for j in range(len(counts.columns)):
            ax.text(j, i, f'{int(counts.iloc[i,j])}/{int(totals.iloc[i,j])}', ha='center', va='center',
                    color='white' if fractions.iloc[i,j] >= .58 else '#183044')
for ax in list(axes.flat)[len(HEATMAP_METHODS):]:
    ax.set_visible(False)
fig.suptitle('Drift: ' + METRIC.replace('_', ' '), fontsize=12)
show_figure(fig, 'drift_sensitivity_grid.png')
save_table(aggregate, 'all_63_settings_aggregate.csv')
save_table(drift_grid_table, 'all_methods_drift_grid.csv', index=True)


## 10  Inspect magnitude sensitivity for all three fault types

Long faults use the previously selected duration; spikes use one reading. Plot three NN alarm rules while retaining the six-method tables. Each point pools three seeds.

In [ ]:
# Inspect magnitude sensitivity for all three fault types
selected_duration = sensitivity[((sensitivity.scenario == 'spike') & (sensitivity.duration_readings == 1)) |
                               ((sensitivity.scenario != 'spike') & (sensitivity.duration_readings == MAIN_DURATION_READINGS))]
amplitude = selected_duration.groupby(['scenario', 'method', 'magnitude_ppm'], as_index=False)[
    ['events', 'detected', 'paired_additional_events']].sum()
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), layout='constrained', sharey=True)
for ax, kind in zip(axes, ['spike', 'drift', 'bias']):
    for method in FOCUS_METHODS:
        part = amplitude[(amplitude.scenario == kind) & (amplitude.method == method)].sort_values('magnitude_ppm')
        ax.plot(part.magnitude_ppm, 100*part.detected/part.events, marker='o',
                label=METHOD_NAMES[method], color=COLOURS[method])
    ax.set(xlabel='Magnitude in ppm', title=kind.title(), ylim=(-3, 103), xticks=[20,40,60])
    ax.grid(alpha=.2)
axes[0].set_ylabel('Event detection (%)')
axes[0].legend(fontsize=8)
show_figure(fig, 'three_fault_magnitude_sensitivity.png')
save_table(amplitude, 'magnitude_sensitivity_all_methods.csv')


## 11  Separate upward and downward drift

Use actual injected counts rather than assuming balanced signs. Direction and background positions can differ, so these results do not prove an algorithm only detects one direction.

In [ ]:
# Separate upward and downward drift
direction_events = sensitivity_events[(sensitivity_events.scenario == 'drift') &
    (sensitivity_events.magnitude_ppm == MAIN_MAGNITUDE_PPM) &
    (sensitivity_events.duration_readings == MAIN_DURATION_READINGS)]
direction_counts = direction_events.groupby(['method', 'sign']).agg(
    events=('event_id', 'size'), detected=('detected', 'sum'), paired_additional=('additional_alarm', 'sum'))
direction_table = pd.DataFrame(index=METHODS)
for sign, label in [(-1, 'downward'), (1, 'upward')]:
    part = direction_counts.xs(sign, level='sign').reindex(METHODS)
    direction_table[label] = [f'{int(a)}/{int(b)}' for a,b in zip(part.detected, part.events)]
    direction_table[label + '_paired_additional'] = [f'{int(a)}/{int(b)}' for a,b in zip(part.paired_additional, part.events)]
direction_table.index = [METHOD_NAMES[m] for m in direction_table.index]
display(direction_table)
save_table(direction_table, 'drift_by_direction.csv', index=True)


## 12  Inspect each default drift event

Each row is one event, with sign, peak absolute residual and cumulative score relative to its limit. A score ratio must exceed 1 to alarm. It is not a CO2 prediction.

In [ ]:
# Inspect each default drift event
def get_event(kind, event_id):
    """Retrieve an existing manifest event; never plan or inject a new fault."""
    found = [e for e in manifest[kind]['events'] if e['event_id'] == event_id]
    if len(found) != 1:
        available = [e['event_id'] for e in manifest[kind]['events']]
        raise ValueError(f'Choose a saved {kind} event ID from {available}.')
    return found[0]

def event_window(kind, event_id, before, after):
    """Return a local saved test window with reading-count context."""
    if not isinstance(before, int) or not isinstance(after, int) or min(before, after) < 0:
        raise ValueError('Context lengths must be nonnegative integers.')
    event = get_event(kind, event_id)
    active = pd.DatetimeIndex(pd.to_datetime(event['active_timestamps_utc'], utc=True))
    index = load_points(kind, 'history_neural').index
    if not active.isin(index).all():
        raise ValueError('Saved event timestamps do not match the saved prediction period.')
    start, end = index.get_loc(active[0]), index.get_loc(active[-1])
    window = index[max(0, start-before):min(len(index), end+after+1)]
    return event, active, window

drift_diagnosis_rows = []
for event in manifest['drift']['events']:
    active = pd.to_datetime(event['active_timestamps_utc'], utc=True)
    nn = load_points('drift', 'history_neural').loc[active]
    old = load_points('drift', 'history_neural_cusum').loc[active]
    new = load_points('drift', 'nn_threshold_cusum').loc[active]
    drift_diagnosis_rows.append({
        'event_id': event['event_id'], 'direction': 'up' if event['sign'] > 0 else 'down',
        'start_utc': str(active[0]), 'duration_minutes': (active[-1]-active[0]).total_seconds()/60,
        'final_offset_ppm': float(nn.observed_co2_ppm.iloc[-1]-nn.original_co2_ppm.iloc[-1]),
        'NN_detected': bool(nn.alarm.any()), 'original_CUSUM_detected': bool(old.alarm.any()),
        'selected_CUSUM_detected': bool(new.alarm.any()),
        'max_abs_NN_residual_ppm': float(nn.residual_ppm.abs().max()),
        'original_score_peak_over_limit': float((old.score/old.threshold).max()),
        'selected_score_peak_over_limit': float((new.score/new.threshold).max())})
drift_diagnosis = pd.DataFrame(drift_diagnosis_rows)
display(drift_diagnosis.round(3))
print('NN point threshold:', float(load_points('drift','history_neural').threshold.iloc[0]), 'ppm')
save_table(drift_diagnosis, 'default_drift_event_diagnosis.csv')


## 13  Plot every distinct prediction around all three fault types

Black is the original record, grey is observed CO2, and four coloured lines are distinct predictions. Red crosses belong only to PLOT_ALARM_METHOD; orange marks injection. Defaults are spike 1, drift 2 and bias 2. **These traces always use default seed 42, 60 ppm and 24-reading long faults; changing common conditions in cell 7 does not change them.** Change IDs, alarm method or context and rerun this cell. CUSUM adds no prediction curves.

In [ ]:
# Plot every distinct prediction around all three fault types
PLOT_EVENT_IDS = {'spike': 1, 'drift': 2, 'bias': 2}
PLOT_ALARM_METHOD = 'nn_threshold_cusum' # Choose a key in METHODS.
POINTS_BEFORE, POINTS_AFTER = 4, 8
if PLOT_ALARM_METHOD not in METHODS:
    raise ValueError(f'Choose one alarm method from {METHODS}.')
fig, axes = plt.subplots(3, 1, figsize=(12, 10), layout='constrained')
local_metrics = []
for ax, kind in zip(axes, ['spike', 'drift', 'bias']):
    event_id = PLOT_EVENT_IDS[kind]
    event, active, window = event_window(kind, event_id, POINTS_BEFORE, POINTS_AFTER)
    nn = load_points(kind, 'history_neural').loc[window]
    ax.plot(window, nn.original_co2_ppm, color='black', lw=2.0, label='Original record before injection', zorder=3)
    ax.plot(window, nn.observed_co2_ppm, color='.60', lw=1.7, label='Observed with simulated faults', zorder=2)
    local = nn[['original_co2_ppm','observed_co2_ppm']].copy()
    for method in PREDICTORS:
        p = load_points(kind, method).loc[window]
        local[method + '_prediction_ppm'] = p.prediction_ppm
        ax.plot(window, p.prediction_ppm, color=COLOURS[method], lw=1.35,
                ls='--' if method in ['change_rule','history_neural'] else '-.',
                label=PREDICTOR_NAMES[method] + ' prediction', zorder=4)
        local_metrics.append({'scenario': kind, 'event_id': event_id, 'predictor': method,
                              'window_readings': len(window),
                              'local_MAE_vs_original_ppm': float((p.prediction_ppm-p.original_co2_ppm).abs().mean())})
    alarm = load_points(kind, PLOT_ALARM_METHOD).loc[window].alarm
    hits = window[alarm.to_numpy()]
    ax.scatter(hits, nn.loc[hits,'observed_co2_ppm'], color='#d62728', marker='x', s=48,
               label='Alarm: ' + METHOD_NAMES[PLOT_ALARM_METHOD], zorder=6)
    if len(active) == 1:
        ax.axvline(active[0], color='#e4a12c', lw=1.3, label='Injected event')
    else:
        ax.axvspan(active[0], active[-1], color='#e4a12c', alpha=.17, label='Injected event')
    locator = mdates.AutoDateLocator(minticks=4, maxticks=7)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator, tz=timezone.utc))
    active_hits = int(alarm.loc[active].sum())
    recovery = pd.to_datetime(event['recovery_timestamps_utc'], utc=True)
    recovery_hits = int(alarm.loc[window.intersection(recovery)].sum())
    ax.set(title=f'{kind.title()} | event {event_id} | active alarms {active_hits} | shown recovery alarms {recovery_hits}',
           ylabel='CO2 (ppm)', xlabel='Time (UTC)')
    ax.grid(alpha=.2)
    local['selected_alarm'] = alarm
    local['active_injection'] = window.isin(active)
    local['recovery'] = window.isin(recovery)
    save_table(local, f'local_{kind}_event{event_id}_predictions.csv', index=True)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='outside upper center', ncol=4, fontsize=8)
show_figure(fig, 'three_faults_all_predictions.png')
local_prediction_quality = pd.DataFrame(local_metrics)
display(local_prediction_quality.round(3))
print('Local MAE describes only the displayed window. Recovery alarms do not rescue a missed active event.')
save_table(local_prediction_quality, 'displayed_window_prediction_MAE.csv')


## 14  Separate NN residuals from the two CUSUM scores

Choose a fault and event ID. Top: residual ppm and NN point limits. Bottom: pre-reset score divided by each rule limit; values above 1 alarm. Resets were handled in the original replay.

In [ ]:
# Separate NN residuals from the two CUSUM scores
TRACE_SCENARIO, TRACE_EVENT_ID = 'drift', 2
TRACE_BEFORE, TRACE_AFTER = 4, 8
if TRACE_SCENARIO not in manifest:
    raise ValueError('Choose spike, drift or bias.')
event, active, window = event_window(TRACE_SCENARIO, TRACE_EVENT_ID, TRACE_BEFORE, TRACE_AFTER)
nn = load_points(TRACE_SCENARIO, 'history_neural').loc[window]
fig, axes = plt.subplots(2, 1, figsize=(11, 6), layout='constrained', sharex=True)
axes[0].plot(window, nn.residual_ppm, color=COLOURS['history_neural'], label='Observed minus NN prediction')
theta = float(nn.threshold.iloc[0])
axes[0].axhline(theta, color='.4', ls=':', label='NN point limits')
axes[0].axhline(-theta, color='.4', ls=':')
axes[0].axhline(0, color='.7', lw=.7)
axes[0].set(ylabel='Residual (ppm)', title=f'{TRACE_SCENARIO.title()} event {TRACE_EVENT_ID}')
trace_export = nn[['observed_co2_ppm','prediction_ppm','residual_ppm']].copy()
for method, marker in [('history_neural_cusum','o'), ('nn_threshold_cusum','x')]:
    p = load_points(TRACE_SCENARIO, method).loc[window]
    ratio = p.score/p.threshold
    axes[1].plot(window, ratio, color=COLOURS[method], label=METHOD_NAMES[method])
    hit_times = window[p.alarm.to_numpy()]
    axes[1].scatter(hit_times, ratio.loc[hit_times], color=COLOURS[method], marker=marker, s=40)
    trace_export[method + '_score_over_limit'] = ratio
    trace_export[method + '_alarm'] = p.alarm
axes[1].axhline(1, color='#b83f38', ls=':', label='CUSUM alarm limit')
axes[1].set(ylabel='Score divided by its limit', xlabel='Time (UTC)')
for ax in axes:
    if len(active) == 1:
        ax.axvline(active[0], color='#e4a12c', lw=1.2)
    else:
        ax.axvspan(active[0], active[-1], color='#e4a12c', alpha=.17)
    ax.grid(alpha=.2)
    ax.legend(fontsize=8, loc='upper left')
locator = mdates.AutoDateLocator(minticks=4, maxticks=8)
axes[1].xaxis.set_major_locator(locator)
axes[1].xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator, tz=timezone.utc))
show_figure(fig, 'nn_residual_and_cusum_scores.png')
save_table(trace_export, 'displayed_nn_cusum_trace.csv', index=True)
old_info = report['models']['history_neural_cusum']
parameter_table = pd.DataFrame([
    ['Original', old_info['k'], old_info['h'], 'accumulated centred residual'],
    ['Selected in equivalent ppm scale', selection['equivalent_k_ppm'], selection['equivalent_h_ppm'], 'accumulated centred residual'],
    ['Selected in threshold-scaled units', selection['kappa'], selection['H'], 'dimensionless accumulated score']],
    columns=['setting','allowance','decision_limit','statistic'])
display(parameter_table.round(4))


## 15  Explain historical contamination with paired predictions

Optionally read the earlier saved NN paired predictions and verify exact agreement with this run. Fault residual = original residual + injected offset - prediction shift. Original values are post-hoc explanation only. If the optional file is missing, this enhancement is skipped.

In [ ]:
# Explain historical contamination with paired predictions
DECOMPOSITION_EVENT_ID = 2
decomposition_summary = pd.DataFrame()
decomposition_source_hash = None
if DECOMPOSITION_FILE is None or not Path(DECOMPOSITION_FILE).is_file():
    print('Optional paired-prediction file unavailable; remaining analysis is still valid.')
else:
    decomposition_source_hash = file_hash(DECOMPOSITION_FILE)
    decomposition = pd.read_csv(DECOMPOSITION_FILE)
    decomposition=decomposition[(decomposition.scenario=='drift')&(decomposition.method=='history_neural')].copy()
    decomposition.index=pd.to_datetime(decomposition.timestamp_utc,utc=True)
    decomposition=decomposition.rename(columns={'original_prediction_ppm':'nn_clean_prediction_ppm',
        'fault_prediction_ppm':'nn_fault_prediction_ppm','fault_residual_ppm':'nn_fault_residual_ppm'})
    decomposition.index = pd.to_datetime(decomposition.index, utc=True)
    current = load_points('drift', 'history_neural').loc[decomposition.index]
    np.testing.assert_allclose(decomposition.nn_fault_prediction_ppm, current.prediction_ppm, rtol=0, atol=1e-10,
                               err_msg='Optional decomposition belongs to a different prediction run.')
    np.testing.assert_allclose(decomposition.observed_co2_ppm, current.observed_co2_ppm, rtol=0, atol=1e-10)
    np.testing.assert_allclose(decomposition.original_co2_ppm, current.original_co2_ppm, rtol=0, atol=1e-10)
    decomposition['original_NN_residual_ppm'] = decomposition.original_co2_ppm-decomposition.nn_clean_prediction_ppm
    decomposition['prediction_shift_ppm'] = decomposition.nn_fault_prediction_ppm-decomposition.nn_clean_prediction_ppm
    reconstructed = decomposition.original_NN_residual_ppm+decomposition.injected_offset_ppm-decomposition.prediction_shift_ppm
    np.testing.assert_allclose(reconstructed, current.residual_ppm, rtol=0, atol=1e-10)
    end = decomposition.groupby('event_id', sort=True).tail(1).copy()
    end['shift_divided_by_injected_offset'] = end.prediction_shift_ppm/end.injected_offset_ppm
    decomposition_summary = end[['event_id','original_co2_ppm','observed_co2_ppm','nn_clean_prediction_ppm',
                                 'nn_fault_prediction_ppm','original_NN_residual_ppm','injected_offset_ppm',
                                 'prediction_shift_ppm','nn_fault_residual_ppm','shift_divided_by_injected_offset']]
    display(decomposition_summary.round(3))
    chosen = end[end.event_id == DECOMPOSITION_EVENT_ID]
    if chosen.empty:
        raise ValueError('Choose a saved default drift event for the decomposition.')
    last = chosen.iloc[0]
    print(f'Event {DECOMPOSITION_EVENT_ID}, final reading: {last.original_NN_residual_ppm:.3f} + '
          f'{last.injected_offset_ppm:.3f} - ({last.prediction_shift_ppm:.3f}) = {last.nn_fault_residual_ppm:.3f} ppm')
    print('The shift ratio is an event-specific response, not feature importance or prediction accuracy.')
    save_table(decomposition_summary, 'nn_drift_prediction_shift.csv', index=True)


## 16  Distinguish event coverage reading coverage and delay

One event hit does not locate every faulty reading. Missed-event delays remain NaN. Paired delay differences use only events detected by both methods; positive means the compared method is later.

In [ ]:
# Distinguish event coverage reading coverage and delay
reading_table = comparison[['scenario','method','events','detected','reading_recall','median_first_alert_delay_minutes']].copy()
reading_table['method'] = reading_table.method.map(METHOD_NAMES)
display(reading_table.round(3))
paired_rows = []
for kind in ['spike','drift','bias']:
    reference = events[(events.scenario == kind) & (events.method == 'history_neural')]
    for method in ['history_neural_cusum','nn_threshold_cusum']:
        other = events[(events.scenario == kind) & (events.method == method)]
        match = reference.merge(other, on=['scenario','event_id'], suffixes=('_NN','_other'), validate='one_to_one')
        both = match.detected_NN & match.detected_other
        difference = match.loc[both,'first_alert_delay_minutes_other']-match.loc[both,'first_alert_delay_minutes_NN']
        paired_rows.append({'scenario':kind,'compared_method':METHOD_NAMES[method], 'events':len(match),
            'both_detected':int(both.sum()), 'NN_only':int((match.detected_NN & ~match.detected_other).sum()),
            'other_only':int((~match.detected_NN & match.detected_other).sum()),
            'neither':int((~match.detected_NN & ~match.detected_other).sum()),
            'median_paired_other_minus_NN_minutes':float(difference.median()) if len(difference) else np.nan})
paired_delay_table = pd.DataFrame(paired_rows)
display(paired_delay_table.round(3))
print('Matched-event delays describe detected subsets. Background burden and misses must also be considered.')
save_table(reading_table, 'event_reading_and_delay_metrics.csv')
save_table(paired_delay_table, 'matched_delays_vs_NN.csv')


## 17  Read recorded computation costs

Original timings include fitting, calibration and median components over four batch replays. Zero additional fitting or prediction for CUSUM means shared NN work, not a free standalone detector. Selected calibration includes development search and is not online latency.

In [ ]:
# Read recorded computation costs
timing_table = timing.set_index('method').reindex(METHODS)[
    ['training_seconds','calibration_seconds','median_prediction_seconds','median_statistic_seconds']].copy()
timing_table['median_prediction_ms'] = 1000*timing_table.pop('median_prediction_seconds')
timing_table['median_statistic_ms'] = 1000*timing_table.pop('median_statistic_seconds')
timing_table['prediction_note'] = ['own reference','own predictor','own predictor','shared NN computed here',
                                  'reuses NN prediction','reuses NN prediction']
timing_table.index = [METHOD_NAMES[m] for m in timing_table.index]
display(timing_table.round(4))
print('Local batch costs exclude output files and plots. They are not universal hardware benchmarks.')
save_table(timing_table, 'recorded_computation_costs.csv', index=True)


## 18  Organise findings for the report

Generate concise findings from the actual tables. Do not optimise every method or assume an overall winner. After changing common conditions, rerun cell 7 onward.

In [ ]:
# Organise findings for the report
def default_count(method, kind):
    """Format a default event count directly from saved results."""
    row = comparison[(comparison.method == method) & (comparison.scenario == kind)].iloc[0]
    return f'{int(row.detected)}/{int(row.events)}'

def pooled_count(method, kind):
    """Format the three-seed count at the current common conditions."""
    row = pooled.loc[(kind, method)]
    return f'{int(row.detected)}/{int(row.events)}'

nn_test = quality[(quality.method == 'history_neural') & (quality.part == 'test_unmodified')].iloc[0]
ols_test = quality[(quality.method == 'history_regression') & (quality.part == 'test_unmodified')].iloc[0]
findings = [
    f'Assess prediction and detection separately: unmodified test MAE is {nn_test.mae_ppm:.3f} ppm for NN and {ols_test.mae_ppm:.3f} ppm for OLS. A small prediction advantage does not establish stronger fault detection.',
    'Default spike / drift / bias counts: NN point ' + ' / '.join(default_count('history_neural',k) for k in ['spike','drift','bias']) +
    '; original CUSUM ' + ' / '.join(default_count('history_neural_cusum',k) for k in ['spike','drift','bias']) +
    '; selected setting ' + ' / '.join(default_count('nn_threshold_cusum',k) for k in ['spike','drift','bias']) + '. This is a trade-off, not improvement on every fault.',
    f'At {MAIN_MAGNITUDE_PPM:g} ppm and {MAIN_DURATION_READINGS} long-fault readings, pooled drift counts across three seeds are NN point ' + pooled_count('history_neural','drift') +
    ', original CUSUM ' + pooled_count('history_neural_cusum','drift') + ', selected setting ' + pooled_count('nn_threshold_cusum','drift') + '. Do not select seeds by performance or treat shared measurements as independent real faults.',
    'First ask whether residuals retain fault evidence, then whether CUSUM reaches its limit. Historical CO2 can move predictions with the offset, while allowances can suppress remaining small sustained residuals.',
    'Unmodified alerts measure background burden, not verified false positives. Recovery alerts do not rescue active-event misses, and event detection does not identify every faulty reading.',
    'Report sequence: compare all six methods, explain key mechanisms, evaluate the completed CUSUM parameter trial, and state conditional conclusions and limits. The completed six-input NN trial is read from notebook 02; strict same-start direction comparisons remain future work.'
]
for number, text in enumerate(findings, 1):
    display(Markdown(f'**{number}.** {text}'))
if SAVE_OUTPUTS:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    (EXPORT_DIR / 'findings.md').write_text('\n\n'.join(f'{n}. {text}' for n,text in enumerate(findings,1)), encoding='utf-8')


## 19  Export the analysis and confirm source integrity

Recheck source hashes and save tables, figures and a summary in the separate analysis directory. Passing means consistent analysis, not that real drift has been solved.

In [ ]:
# Export the analysis and confirm source integrity
current_files = sorted(p for p in RUN_DIR.rglob('*') if p.is_file())
hashes_after = {str(p.relative_to(RUN_DIR)): file_hash(p) for p in current_files}
assert hashes_after == SOURCE_HASHES, 'Source experiment files changed during analysis.'
if decomposition_source_hash is not None:
    assert file_hash(DECOMPOSITION_FILE) == decomposition_source_hash
integrity = {
    'source_files_unchanged': True,
    '63_cases_and_378_detector_rows': len(cases) == 63 and len(sensitivity) == 378,
    'three_NN_alarm_rules_share_predictions': True,
    'aggregates_match_saved_experiment': True,
    'optional_decomposition_verified': decomposition_source_hash is not None,
    'no_model_fitting_or_fault_injection': True}
display(pd.DataFrame(integrity.items(), columns=['check','result']))
summary = {
    'analysis_version':'methods_analysis_v1', 'source_run':str(RUN_DIR),
    'study_status':report['status'], 'sensor_id':report['audit']['sensor_id'],
    'main_magnitude_ppm':MAIN_MAGNITUDE_PPM, 'main_duration_readings':MAIN_DURATION_READINGS,
    'source_hashes':SOURCE_HASHES, 'integrity':integrity,
    'default_comparison':json.loads(comparison.drop(columns='name').to_json(orient='records')),
    'pooled_common_conditions':json.loads(pooled.reset_index().to_json(orient='records')),
    'drift_diagnosis':json.loads(drift_diagnosis.to_json(orient='records')),
    'findings':findings,
    'limits':['Exploratory on an inspected period','No verified healthy or real-fault labels',
              'Shared measurements across simulations','Reading duration can change event positions']}
if SAVE_OUTPUTS:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    (EXPORT_DIR / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    save_table(sensitivity, 'all_378_detector_settings.csv')
    print('Analysis completed. Tables and figures:', EXPORT_DIR)
else:
    print('Analysis completed without exporting files.')


## How to use the analysis

1. Start with cells 5–8 to distinguish prediction error, default detection, pooled seeds and seed variability.
2. Use cells 9–11 for magnitude, duration and direction sensitivity, retaining raw and paired additional counts.
3. Cells 12–15 explain mechanisms; rerun cell 13 to change local events and the selected alarm rule.
4. Cells 16–19 inspect coverage, delay and computation and export report material.

To retrain or inject new magnitudes or placements, use the computational trial notebook. This file analyses saved results only. Notebook 02 contains the completed input comparison, which is read in the final two analysis cells. Keep measurements and executed outputs local.

## Placement and magnitude evidence used by the report

These panels pool the three injection seeds with a fixed long-fault duration of 24 readings. Whiskers are the observed seed range, not confidence intervals. All six detectors remain in the tables; three rules sharing NN predictions are shown in the figure.

In [ ]:
from co2_project.analysis import magnitude_summary, plot_magnitude_sensitivity, seed_table
amplitude_summary=magnitude_summary(sensitivity)
display(amplitude_summary)
save_table(amplitude_summary,'all_methods_magnitude_raw_and_added.csv')
placement_table=seed_table(sensitivity)
display(placement_table)
save_table(placement_table,'all_methods_seed60.csv')
fig=plot_magnitude_sensitivity(sensitivity)
show_figure(fig,'report_magnitude_and_placement.png')
print('Use raw and paired additional detections together. Shared records are not independent real faults.')


## Completed input comparison and its counterexample

Removing history prevents a CO2-only injection from moving prediction, but a wider residual distribution can enlarge validation-calibrated limits. The exception at 20 ppm over 48 readings is retained rather than selecting only favourable conditions.

In [ ]:
input_quality=pd.read_csv(INPUT_RUN_DIR/'prediction_quality.csv')
input_default=pd.read_csv(INPUT_RUN_DIR/'controlled_default.csv')
input_grid=pd.read_csv(INPUT_RUN_DIR/'sensitivity_aggregate.csv')
display(input_quality.round(3))
display(input_default[['scenario','method','detected','events','paired_additional_events','unmodified_test_alert_rate']])
display(input_grid[(input_grid.scenario=='drift')&(input_grid.magnitude_ppm==20)&(input_grid.duration_readings==48)&input_grid.method.isin(['history_neural_cusum','environment_neural_cusum'])])
display(pd.read_csv(INPUT_RUN_DIR/'event_end_decomposition.csv').query("scenario=='drift' and event_id==2").round(3))
save_table(input_quality,'input_comparison_quality.csv')
save_table(input_default,'input_comparison_default.csv')
print('NN training randomness and an independent future period were not evaluated.')
